In [1]:
import pandas as pd

In [2]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")


In [4]:
print("Training data shape:", train_df.shape)
print("Testing data shape:", test_df.shape)
print("\nTraining columns:")
print(train_df.columns.tolist())
print("\nTesting columns:")
print(test_df.columns.tolist())



Training data shape: (159571, 8)
Testing data shape: (153164, 2)

Training columns:
['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

Testing columns:
['id', 'comment_text']


In [ ]:
train_df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [6]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 159571 entries, 0 to 159570
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   id             159571 non-null  str  
 1   comment_text   159571 non-null  str  
 2   toxic          159571 non-null  int64
 3   severe_toxic   159571 non-null  int64
 4   obscene        159571 non-null  int64
 5   threat         159571 non-null  int64
 6   insult         159571 non-null  int64
 7   identity_hate  159571 non-null  int64
dtypes: int64(6), str(2)
memory usage: 72.2 MB


In [7]:
train_df.describe()

,toxic,severe_toxic,obscene,threat,insult,identity_hate
count,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000
mean,0.095844,0.009996,0.052948,0.002996,0.049364,0.008805
std,0.294379,0.099477,0.223931,0.054650,0.216627,0.093420
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [83]:
test_df.head()

,id,comment_text
0,00001cee341fdb12,Yo bitch Ja Rule is more succesful then you'll...
1,0000247867823ef7,== From RfC == \n\n The title is fine as it is...
2,00013b17ad220c46,""" \n\n == Sources == \n\n * Zawe Ashton on Lap..."
3,00017563c3f7919a,":If you have a look back at the source, the in..."
4,00017695ad8997eb,I don't anonymously edit articles at all.


#####  SELECT FEATURES AND TARGETS



In [3]:
target_columns = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

train_df[target_columns].sum()

toxic            15294
severe_toxic      1595
obscene           8449
threat             478
insult            7877
identity_hate     1405
dtype: int64

In [10]:
train_df[target_columns].mean() * 100

toxic            9.584448
severe_toxic     0.999555
obscene          5.294822
threat           0.299553
insult           4.936361
identity_hate    0.880486
dtype: float64

In [4]:
# Input feature
X = train_df["comment_text"]

# Target labels
y = train_df[target_columns]


In [5]:
print(X.shape)
print(y.shape)

(159571,)
(159571, 6)


##### Clean the Text

In [6]:
import re

In [7]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [8]:
X_clean = X.apply(clean_text)

In [9]:
print(X.iloc[1])
print("\n")
print(X_clean.iloc[1])

D'aww! He matches this background colour I'm seemingly stuck with. Thanks.  (talk) 21:51, January 11, 2016 (UTC)


daww he matches this background colour im seemingly stuck with thanks talk january utc


##### Remove stopwords


In [14]:
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")

stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    words = text.split()
    return " ".join(
        word for word in words
        if word not in stop_words
    )

X_no_stopwords = X_clean.apply(remove_stopwords)

print("Before:")
print(X_clean.iloc[1])

print("\nAfter:")
print(X_no_stopwords.iloc[1])

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Devendra\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Before:
daww he matches this background colour im seemingly stuck with thanks talk january utc

After:
daww matches background colour im seemingly stuck thanks talk january utc


#### Tokenization

In [10]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [15]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [16]:
tokenizer = Tokenizer(num_words=20000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_no_stopwords)

In [17]:
X_sequences = tokenizer.texts_to_sequences(X_no_stopwords)

In [18]:
print(X_sequences[0])

[524, 45, 48, 514, 4394, 11194, 921, 209, 1885, 10644, 6313, 2480, 2634, 38, 1010, 15000, 2667, 6, 10, 137, 301, 5, 3, 59, 14, 3211]


In [19]:
len(X_sequences)

159571

##### Padding

In [20]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [21]:
X_padded = pad_sequences(
    X_sequences,
    maxlen=200,
    padding="post",
    truncating="post"
)

In [22]:
print(X_padded.shape)

(159571, 200)


#### Train/Validation Split

In [23]:
from sklearn.model_selection import train_test_split


In [24]:
X_train, X_val, y_train, y_val = train_test_split(
    X_padded,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train.shape)
print("Validation data:", X_val.shape)

Training data: (127656, 200)
Validation data: (31915, 200)


In [25]:
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

X_train: (127656, 200)
X_val: (31915, 200)
y_train: (127656, 6)
y_val: (31915, 6)


#### Build the Deep Learning Model

In [27]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense

#LSTM
model = Sequential([
    Input(shape=(200,)),
    Embedding(input_dim=20000, output_dim=128),
    LSTM(64),
    Dense(32, activation="relu"),
    Dense(6, activation="sigmoid")
])

##  Compile
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 200, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,611,686 (9.96 MB)

 Trainable params: 2,611,686 (9.96 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
X_train.shape

(127656, 200)

##### TRAIN MODEL


In [29]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=64
)

Epoch 1/5
1995/1995 ━━━━━━━━━━━━━━━━━━━━ 178s 88ms/step - accuracy: 0.9831 - loss: 0.1429 - val_accuracy: 0.9939 - val_loss: 0.1347
Epoch 2/5
1995/1995 ━━━━━━━━━━━━━━━━━━━━ 182s 91ms/step - accuracy: 0.9936 - loss: 0.0610 - val_accuracy: 0.9938 - val_loss: 0.0531
Epoch 3/5
1995/1995 ━━━━━━━━━━━━━━━━━━━━ 152s 76ms/step - accuracy: 0.9939 - loss: 0.0457 - val_accuracy: 0.9933 - val_loss: 0.0517
Epoch 4/5
1995/1995 ━━━━━━━━━━━━━━━━━━━━ 179s 90ms/step - accuracy: 0.9938 - loss: 0.0406 - val_accuracy: 0.9941 - val_loss: 0.0547
Epoch 5/5
1995/1995 ━━━━━━━━━━━━━━━━━━━━ 177s 89ms/step - accuracy: 0.9934 - loss: 0.0364 - val_accuracy: 0.9940 - val_loss: 0.0582


##### Check Training Performance

In [30]:
print("Training Accuracy:", history.history["accuracy"][-1])
print("Validation Accuracy:", history.history["val_accuracy"][-1])

print("Training Loss:", history.history["loss"][-1])
print("Validation Loss:", history.history["val_loss"][-1])

Training Accuracy: 0.9934041500091553
Validation Accuracy: 0.994046688079834
Training Loss: 0.03636091575026512
Validation Loss: 0.05818991735577583


##### MODEL EVALUATION



In [81]:
y_pred_prob = model.predict(X_val)

y_pred = (y_pred_prob >= 0.5).astype(int)

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        y_pred,
        target_names=target_columns
    )
)

998/998 ━━━━━━━━━━━━━━━━━━━━ 16s 16ms/step

Classification Report:
               precision    recall  f1-score   support

        toxic       0.79      0.75      0.77      3056
 severe_toxic       0.59      0.08      0.15       321
      obscene       0.78      0.79      0.79      1715
       threat       0.00      0.00      0.00        74
       insult       0.70      0.65      0.67      1614
identity_hate       0.00      0.00      0.00       294

    micro avg       0.77      0.67      0.71      7074
    macro avg       0.48      0.38      0.40      7074
 weighted avg       0.72      0.67      0.68      7074
  samples avg       0.06      0.06      0.06      7074



c:\Users\Devendra\anaconda3\envs\tensorflow_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Devendra\anaconda3\envs\tensorflow_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Devendra\anaconda3\envs\tensorflow_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.cap

##### FINAL THRESHOLDS

In [82]:
thresholds = [
    0.5,   # toxic
    0.5,   # severe_toxic
    0.5,   # obscene
    0.1,   # threat
    0.5,   # insult
    0.1    # identity_hate
]

#####  FINAL EVALUATION WITH THRESHOLDS


In [83]:
y_pred_final = np.zeros_like(
    y_pred_prob,
    dtype=int
)

for i, threshold in enumerate(thresholds):
    y_pred_final[:, i] = (
        y_pred_prob[:, i] >= threshold
    ).astype(int)

print("\nFinal Classification Report:")
print(
    classification_report(
        y_val,
        y_pred_final,
        target_names=target_columns
    )
)



Final Classification Report:
               precision    recall  f1-score   support

        toxic       0.79      0.75      0.77      3056
 severe_toxic       0.59      0.08      0.15       321
      obscene       0.78      0.79      0.79      1715
       threat       0.11      0.01      0.02        74
       insult       0.70      0.65      0.67      1614
identity_hate       0.16      0.52      0.24       294

    micro avg       0.68      0.69      0.68      7074
    macro avg       0.52      0.47      0.44      7074
 weighted avg       0.73      0.69      0.69      7074
  samples avg       0.06      0.06      0.06      7074



c:\Users\Devendra\anaconda3\envs\tensorflow_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Devendra\anaconda3\envs\tensorflow_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Devendra\anaconda3\envs\tensorflow_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{met

##### SAVE MODEL + TOKENIZER


In [84]:
model.save("toxicity_lstm_model.keras")

with open("tokenizer.pkl", "wb") as file:
    pickle.dump(tokenizer, file)

print("\n✅ Model saved")
print("✅ Tokenizer saved")


✅ Model saved
✅ Tokenizer saved


##### PREDICTION FUNCTION



In [85]:
def predict_toxicity(comment):

    comment = clean_text(comment)
    comment = remove_stopwords(comment)

    sequence = tokenizer.texts_to_sequences([comment])

    padded = pad_sequences(
        sequence,
        maxlen=200,
        padding="post",
        truncating="post"
    )

    probabilities = model.predict(
        padded,
        verbose=0
    )[0]

    return probabilities

##### TEST ONE COMMENT


In [86]:

result = predict_toxicity(
    "You are a stupid and disgusting person"
)

print("\nPrediction probabilities:")
print(result)

print("\nPrediction results:")

for label, probability, threshold in zip(
    target_columns,
    result,
    thresholds
):

    prediction = (
        "YES"
        if probability >= threshold
        else "NO"
    )

    print(
        f"{label}: "
        f"{probability:.4f} → {prediction}"
    )




Prediction probabilities:
[0.9625722  0.04010306 0.78458446 0.03294602 0.62286675 0.08711843]

Prediction results:
toxic: 0.9626 → YES
severe_toxic: 0.0401 → NO
obscene: 0.7846 → YES
threat: 0.0329 → NO
insult: 0.6229 → YES
identity_hate: 0.0871 → NO


#####  TEST DATA PREDICTION


In [87]:
X_test = test_df["comment_text"]

X_test_processed = (
    X_test
    .apply(clean_text)
    .apply(remove_stopwords)
)

X_test_sequences = tokenizer.texts_to_sequences(
    X_test_processed
)

X_test_padded = pad_sequences(
    X_test_sequences,
    maxlen=200,
    padding="post",
    truncating="post"
)

test_pred_prob = model.predict(
    X_test_padded,
    batch_size=64,
    verbose=1
)

print("\nTest prediction shape:",
      test_pred_prob.shape)

2394/2394 ━━━━━━━━━━━━━━━━━━━━ 83s 35ms/step

Test prediction shape: (153164, 6)


#####  CREATE FINAL CSV

In [88]:
test_pred = np.zeros_like(
    test_pred_prob,
    dtype=int
)

for i, threshold in enumerate(thresholds):
    test_pred[:, i] = (
        test_pred_prob[:, i] >= threshold
    ).astype(int)

test_predictions = pd.DataFrame(
    test_pred,
    columns=target_columns
)

test_predictions.insert(
    0,
    "id",
    test_df["id"].values
)

test_predictions.to_csv(
    "test_predictions.csv",
    index=False
)

print("\n✅ test_predictions.csv created!")
print("Shape:", test_predictions.shape)
print(test_predictions.head())


✅ test_predictions.csv created!
Shape: (153164, 7)
                 id  toxic  severe_toxic  obscene  threat  insult  \
0  00001cee341fdb12      1             0        1       0       1   
1  0000247867823ef7      0             0        0       0       0   
2  00013b17ad220c46      0             0        0       0       0   
3  00017563c3f7919a      0             0        0       0       0   
4  00017695ad8997eb      0             0        0       0       0   

   identity_hate  
0              1  
1              0  
2              0  
3              0  
4              0  


##### Positive comment

In [89]:
test_comment1 = "I really like this project and your work"

result1 = predict_toxicity(test_comment1)

print("\nPositive Comment:")
print(test_comment1)
print("Prediction Probabilities:")
print(result1)


Positive Comment:
I really like this project and your work
Prediction Probabilities:
[3.1931400e-02 3.3182980e-05 2.6763044e-03 1.9869893e-03 6.3439482e-03
 2.6282195e-03]


##### Toxic comment

In [90]:
test_comment2 = "You are a stupid and disgusting person"

result2 = predict_toxicity(test_comment2)

print("\nToxic Comment:")
print(test_comment2)
print("Prediction Probabilities:")
print(result2)


Toxic Comment:
You are a stupid and disgusting person
Prediction Probabilities:
[0.9625722  0.04010306 0.78458446 0.03294602 0.62286675 0.08711843]


In [91]:
def show_prediction(comment):

    probabilities = predict_toxicity(comment)

    print("\nComment:")
    print(comment)

    print("\nResults:")

    for label, probability, threshold in zip(
        target_columns,
        probabilities,
        thresholds
    ):
        prediction = (
            "YES"
            if probability >= threshold
            else "NO"
        )

        print(
            f"{label}: "
            f"{probability:.4f} → {prediction}"
        )


show_prediction(
    "You are a stupid and disgusting person"
)


Comment:
You are a stupid and disgusting person

Results:
toxic: 0.9626 → YES
severe_toxic: 0.0401 → NO
obscene: 0.7846 → YES
threat: 0.0329 → NO
insult: 0.6229 → YES
identity_hate: 0.0871 → NO
